# IQM CLOPS Calibration

Runs IQM's own official CLOPS (Circuit Layer Operations Per Second) benchmark against real IQM machines, using their `iqm-benchmarks` package (`iqm.benchmarks.quantum_volume.clops.CLOPSBenchmark`).

**Why this exists:** IQM only publicly documents CLOPS for Garnet. `estimate_qpu_time_clops()` (the IBM-side model in `quantum/hardware/qpu_time_estimate.py`) needs a published CLOPS number to work at all — for any other IQM machine, or to cross-check Garnet's documented figure, this notebook is how you get one. See `quantum/hardware/README.md`'s "IQM CLOPS per machine" section for the full context and why we reimplement `iqm_two_qubit_depth` ourselves but run *this* benchmark via IQM's real library rather than recreating the protocol (a subtly-wrong reimplementation would silently stop being comparable to IQM's own numbers, which defeats the point).

**Covers multiple machines, run independently.** The two production-relevant tiers (`IQM_MACHINE_TIERS` in `Pennylane_solver.py`) are Garnet and Emerald — each gets its own section below, callable on its own schedule rather than forced into one all-or-nothing run. Sirius is intentionally left out for now (see section 2) but adding it later is just extending the machine list there plus one more `## Run — <machine>` section.

**This is a one-off calibration exercise, not part of the solve path** — deliberately kept out of `pyproject.toml` (see the install cell below) so its dependencies (`iqm-benchmarks`, and transitively `matplotlib`/`xarray`) never load for a normal `spooky-solve` run.

**Save this notebook with its output cells intact after running** — the whole point is a durable, inspectable record of what was measured, when, and against what qubit layout, not just a number someone has to trust.

## 0. Install dependencies

Self-contained in this notebook — not a `spooky` dependency. Installs into whatever kernel is running this notebook.

In [6]:
!uv pip install -q iqm-benchmarks python-dotenv


## 1. Credentials

Same convention as the rest of this project (`Pennylane_solver.py`, `quantum/prueba/iqm/qpu_test.py`): `IQM_TOKEN` read from `.env` via `python-dotenv`, not hardcoded here.

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("IQM_TOKEN"), "IQM_TOKEN not found — set it in .env before continuing"

IQM_URL = "https://resonance.iqm.tech"  # matches Pennylane_solver.py's IQM_URL


## 2. Machines to calibrate

Uses **all qubits on each machine** — Garnet's 20, Emerald's 54 — for a full-device CLOPS number, not a hand-picked subset. `CLOPSConfiguration.qubits` sets both qubit count and circuit depth (`depth = len(qubits)`), so more qubits means deeper circuits per shot (same 100,000 total shots either way, but each one takes longer to execute) — that's the actual cost lever here, not which specific qubits are chosen.

`QUBIT_LAYOUTS` below is derived automatically from each machine's live `backend.num_qubits` — no manual selection needed. The coupling map and per-edge two-qubit gate error printed alongside it are for sanity-checking the device's current calibration state before spending real quota, not something you need to act on.

In [2]:
from iqm.qiskit_iqm import IQMProvider

QUBIT_LAYOUTS = {}

for machine in ["garnet", "emerald"]:
    # "sirius" not included yet: Qrisp's IQM connector can't handle its
    # resonator-mediated coupling (see IQM_MACHINE_TIERS in
    # Pennylane_solver.py) — but that's a Qrisp-specific limitation, not
    # one that applies to this notebook's iqm.qiskit_iqm path. Worth
    # trying once you're ready; just add "sirius" to this list.
    os.environ["USE_TIMESLOT"] = "False"
    # use_metrics=True: without it, InstructionProperties (including
    # two-qubit gate error below) are not populated at all.
    backend = IQMProvider(IQM_URL, quantum_computer=machine).get_backend(use_metrics=True)
    QUBIT_LAYOUTS[machine] = list(range(backend.num_qubits))

    print(f"{machine}: {backend.num_qubits} qubits -> using all of them")
    print("Coupling map (physical qubit index pairs):")
    print(list(backend.coupling_map.get_edges()))
    print("Two-qubit gate error per edge (for reference, not selection):")
    for q1, q2 in backend.coupling_map.get_edges():
        if q1 < q2:  # coupling map lists both directions; only print once per edge
            # Some pairs have no reported InstructionProperties at all (None,
            # not just a None .error) — try both orderings, then fall back to N/A
            # rather than crash; cz is symmetric so an untouched direction is fine.
            props = backend.target["cz"].get((q1, q2)) or backend.target["cz"].get((q2, q1))
            error = props.error if props is not None else None
            error_str = f"{error:.4f}" if error is not None else "N/A"
            print(f"  ({q1}, {q2}): {error_str}")
    print()


/home/javideus/.venvs/quantum/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


garnet: 20 qubits -> using all of them
Coupling map (physical qubit index pairs):
[(1, 0), (1, 4), (3, 0), (3, 2), (3, 4), (3, 8), (5, 4), (5, 6), (5, 10), (7, 2), (7, 8), (7, 12), (9, 4), (9, 8), (9, 10), (9, 14), (11, 6), (11, 10), (11, 16), (13, 8), (13, 12), (13, 14), (13, 17), (15, 10), (15, 14), (15, 16), (15, 19), (18, 14), (18, 17), (18, 19)]
Two-qubit gate error per edge (for reference, not selection):
  (1, 4): 0.0088
  (3, 4): 0.0053
  (3, 8): 0.0056
  (5, 6): 0.0115
  (5, 10): 0.0090
  (7, 8): 0.0038
  (7, 12): 0.0048
  (9, 10): 0.0133
  (9, 14): 0.0134
  (11, 16): 0.0022
  (13, 14): 0.0094
  (13, 17): 0.0046
  (15, 16): 0.0049
  (15, 19): 0.0039
  (18, 19): 0.0028



Missing duration for cz.slepian_crf.QB50__QB51
Missing fidelity for cz.slepian_crf.QB50__QB51


emerald: 54 qubits -> using all of them
Coupling map (physical qubit index pairs):
[(0, 1), (0, 4), (3, 2), (3, 4), (3, 9), (5, 1), (5, 4), (5, 6), (5, 11), (8, 2), (8, 7), (8, 9), (8, 16), (10, 4), (10, 9), (10, 11), (10, 18), (12, 6), (12, 11), (12, 13), (12, 20), (15, 7), (15, 14), (15, 16), (15, 23), (17, 9), (17, 16), (17, 18), (17, 25), (19, 11), (19, 18), (19, 20), (19, 27), (21, 13), (21, 20), (21, 29), (22, 23), (24, 16), (24, 23), (24, 25), (24, 32), (26, 18), (26, 25), (26, 27), (26, 34), (28, 20), (28, 27), (28, 29), (28, 36), (30, 29), (30, 38), (31, 32), (31, 39), (33, 25), (33, 32), (33, 34), (33, 41), (35, 27), (35, 34), (35, 36), (35, 43), (37, 29), (37, 36), (37, 38), (37, 45), (40, 32), (40, 39), (40, 41), (40, 46), (42, 34), (42, 41), (42, 43), (42, 48), (43, 49), (44, 36), (44, 43), (44, 45), (44, 50), (47, 41), (47, 46), (47, 48), (47, 51), (49, 50), (52, 48), (52, 51), (52, 53)]
Two-qubit gate error per edge (for reference, not selection):
  (0, 1): 0.0023
  (0, 

## 3. ⚠️ Cost warning — read before running any calibration cell below

Default parameters submit **100 circuits × 10 updates × 100 shots = 100,000 shots per machine** against real hardware. Calibrating both Garnet and Emerald is 200,000 shots total — a substantially larger real-money/quota spend than the estimation/calibration runs elsewhere in this project (500–10,000 shots). Run one machine's section at a time and decide deliberately, rather than executing the whole notebook top-to-bottom on autopilot. Using the full device (section 2) means deeper circuits too — Emerald's 54-qubit run will take meaningfully longer per shot than Garnet's 20-qubit one.

**Observed in practice (2026-08-22, Garnet):** ~52s of QPU time, ~26 credits — against a 30 credits/month allotment, that's essentially the *entire* monthly budget for one run. This is not a notebook to run repeatedly or casually; treat each execution as a deliberate, budgeted spend, not a quick check. Emerald was also unavailable (down) at this time, separately from the credit question — worth confirming availability before planning a run around it.

## 4. Calibration function

One call per machine: connects, configures, runs the benchmark, analyzes it, and appends the result to `results/iqm_clops.json` (self-contained JSON write — deliberately not importing from `quantum.hardware`, since this notebook may be launched from a different working directory than a normal `spooky-solve` run). Returns `(run_clops, result_clops)` so the calling cell can display observations and plots for that specific machine.

Called separately per machine below rather than in a loop, so each machine's output stays in its own clearly-labeled cell block — useful when machines are calibrated on different days, and when reviewing this notebook's saved output later.

In [3]:
import datetime
import json
from pathlib import Path

from iqm.benchmarks.quantum_volume.clops import CLOPSBenchmark, CLOPSConfiguration

OUTPUT_PATH = Path("../../results/iqm_clops.json")  # relative to quantum/hardware/


def calibrate_machine(machine, qubits):
    os.environ["USE_TIMESLOT"] = "False"  # pay-as-you-go, not a reserved timeslot booking
    provider = IQMProvider(IQM_URL, quantum_computer=machine)
    backend = provider.get_backend(use_metrics=True)
    print(f"Connected to {machine}: {backend.num_qubits} qubits")

    config = CLOPSConfiguration(
        qubits=qubits,
        num_circuits=100,   # by definition, per CLOPSConfiguration's docstring
        num_updates=10,     # by definition
        num_shots=100,      # by definition
        max_circuits_per_batch=100,
    )

    run_timestamp = datetime.datetime.now(datetime.timezone.utc).isoformat()
    benchmark = CLOPSBenchmark(backend, config)
    run_clops = benchmark.run()
    result_clops = benchmark.analyze()

    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    store = json.loads(OUTPUT_PATH.read_text()) if OUTPUT_PATH.exists() else {}
    store.setdefault(machine, []).append(
        {
            "date": run_timestamp,
            "qubits": qubits,
            "num_circuits": config.num_circuits,
            "num_updates": config.num_updates,
            "num_shots": config.num_shots,
            "clops_v": next(
                (o.value for o in result_clops.observations if o.name == "clops_v"), None
            ),
            "clops_h": next(
                (o.value for o in result_clops.observations if o.name == "clops_h"), None
            ),
        }
    )
    OUTPUT_PATH.write_text(json.dumps(store, indent=2))
    print(f"Appended to {OUTPUT_PATH}")

    return run_clops, result_clops


## 5. Run — Garnet

In [4]:
run_garnet, result_garnet = calibrate_machine("garnet", QUBIT_LAYOUTS["garnet"])


2026-08-22 23:19:00,187 - iqm.benchmarks.logging_config - INFO - NB: CLOPS should be estimated with same qubit layout and optional inputs used to establish QV!
2026-08-22 23:19:00,188 - iqm.benchmarks.logging_config - INFO - Now generating 100 parametrized circuit templates on qubits [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]


Connected to garnet: 20 qubits


2026-08-22 23:19:05,852 - iqm.benchmarks.logging_config - INFO - Will transpile all 100 circuits according to "fixed" physical layout
2026-08-22 23:19:05,854 - iqm.benchmarks.logging_config - INFO - Transpiling for backend garnet with optimization level 3, sabre routing method including SQG optimization all circuits
2026-08-22 23:20:14,293 - iqm.benchmarks.logging_config - INFO - CLOPS time started
2026-08-22 23:20:14,293 - iqm.benchmarks.logging_config - INFO - Update 1/10
2026-08-22 23:20:14,294 - iqm.benchmarks.logging_config - INFO - Assigning random parameters to all 100 circuits
2026-08-22 23:20:35,856 - iqm.benchmarks.logging_config - INFO - Executing the corresponding circuit batch
2026-08-22 23:20:35,856 - iqm.benchmarks.logging_config - INFO - Submitting batch with 100 circuits corresponding to qubits [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
2026-08-22 23:20:35,857 - iqm.benchmarks.logging_config - INFO - max_circuits_per_batch restriction: submi

KeyboardInterrupt: 

In [ ]:
for obs in result_garnet.observations:
    print(obs)
run_garnet.dataset.attrs["operation_counts"]


In [ ]:
result_garnet.plot_all()


## 6. Run — Emerald

In [ ]:
run_emerald, result_emerald = calibrate_machine("emerald", QUBIT_LAYOUTS["emerald"])


2026-08-22 23:01:41,154 - iqm.station_control.client.qon - WARNING - Missing duration for cz.slepian_crf.QB50__QB51
2026-08-22 23:01:41,157 - iqm.station_control.client.qon - WARNING - Missing fidelity for cz.slepian_crf.QB50__QB51
2026-08-22 23:01:41,326 - iqm.benchmarks.logging_config - INFO - NB: CLOPS should be estimated with same qubit layout and optional inputs used to establish QV!
2026-08-22 23:01:41,327 - iqm.benchmarks.logging_config - INFO - Now generating 100 parametrized circuit templates on qubits [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53]


Connected to emerald: 54 qubits


2026-08-22 23:02:04,973 - iqm.benchmarks.logging_config - INFO - Will transpile all 100 circuits according to "fixed" physical layout
2026-08-22 23:02:04,974 - iqm.benchmarks.logging_config - INFO - Transpiling for backend emerald with optimization level 3, sabre routing method including SQG optimization all circuits
2026-08-22 23:12:44,196 - iqm.benchmarks.logging_config - INFO - CLOPS time started
2026-08-22 23:12:44,199 - iqm.benchmarks.logging_config - INFO - Update 1/10
2026-08-22 23:12:44,200 - iqm.benchmarks.logging_config - INFO - Assigning random parameters to all 100 circuits
2026-08-22 23:16:25,101 - iqm.benchmarks.logging_config - INFO - Executing the corresponding circuit batch
2026-08-22 23:16:25,127 - iqm.benchmarks.logging_config - INFO - Submitting batch with 100 circuits corresponding to qubits [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 4

In [ ]:
for obs in result_emerald.observations:
    print(obs)
run_emerald.dataset.attrs["operation_counts"]


In [ ]:
result_emerald.plot_all()


## 7. Adding more machines later

1. Add the machine name to the `for machine in [...]` loop in section 2 (it'll auto-derive its qubit layout the same way).
2. Add a new `## Run — <machine>` section following the exact three-cell pattern used for Garnet/Emerald above (call `calibrate_machine`, print observations, `plot_all()`).

Existing results aren't affected — `calibrate_machine` appends to `results/iqm_clops.json` per machine, so re-running this notebook later (a new machine, or a recalibration of an existing one) only adds data points.

### Results summary — fill in after running

| Machine | Date | Qubits used | CLOPS_V | CLOPS_H |
|---|---|---|---|---|
| Garnet | *TODO* | *TODO (all, from section 2)* | *TODO* | *TODO* |
| Emerald | *TODO* | *TODO (all, from section 2)* | *TODO* | *TODO* |
| Sirius | — | — | — | not yet attempted (see section 2) |

*A sentence or two here on anything notable — e.g. how CLOPS_V compares to Garnet's publicly documented figure, how the two machines compare to each other, or anything unusual in either machine's elapsed-time breakdown plot.*

## Next steps

- Update `quantum/hardware/README.md`: turn the "IQM CLOPS per machine" bullet in "Ideas for later" into a real findings section (mirroring how "IBM quota: what we've learned from real runs" documents actual measured numbers), linking back to this notebook.
- `results/iqm_clops.json` accumulates every machine and every re-run — a future `get_iqm_clops(machine_name)` reader (mirroring `qpu_clops.py`'s `get_backend_clops()` for IBM) could read from it, though that reader doesn't exist yet — not built until there's an actual consumer for it, same reasoning as everywhere else in this project's calibration work.